## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        initializer = tf.initializers.TruncatedNormal(stddev=0.1)

        self.W1 = tf.Variable(initializer(shape=[784, 100]))
        self.b1 = tf.Variable(tf.zeros([100]))

        self.W2 = tf.Variable(initializer(shape=[100, 10]))
        self.b2 = tf.Variable(tf.zeros([10]))
    def __call__(self, x):
        x = tf.reshape(x, [-1, 784])
        
        h1 = tf.matmul(x, self.W1) + self.b1
        h1_relu = tf.nn.relu(h1)
        
        logits = tf.matmul(h1_relu, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.4396636 ; accuracy 0.07056667
epoch 1 : loss 2.430327 ; accuracy 0.07321667
epoch 2 : loss 2.4212143 ; accuracy 0.07563333
epoch 3 : loss 2.4123075 ; accuracy 0.07808334
epoch 4 : loss 2.4035907 ; accuracy 0.0801
epoch 5 : loss 2.3950498 ; accuracy 0.083116665
epoch 6 : loss 2.3866744 ; accuracy 0.0857
epoch 7 : loss 2.3784523 ; accuracy 0.088766664
epoch 8 : loss 2.370373 ; accuracy 0.09198333
epoch 9 : loss 2.362426 ; accuracy 0.0954
epoch 10 : loss 2.3546042 ; accuracy 0.09858333
epoch 11 : loss 2.346898 ; accuracy 0.101966664
epoch 12 : loss 2.3393025 ; accuracy 0.10545
epoch 13 : loss 2.3318107 ; accuracy 0.10905
epoch 14 : loss 2.3244162 ; accuracy 0.1132
epoch 15 : loss 2.3171144 ; accuracy 0.117
epoch 16 : loss 2.3098986 ; accuracy 0.12078334
epoch 17 : loss 2.3027656 ; accuracy 0.124916665
epoch 18 : loss 2.29571 ; accuracy 0.12873334
epoch 19 : loss 2.2887275 ; accuracy 0.13308333
epoch 20 : loss 2.2818136 ; accuracy 0.13791667
epoch 21 : loss 2.2749655 ; acc